In [5]:
!g++ --version

g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [2]:
%pip install -q duckdb==1.5.5 requests

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin

import duckdb
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ----------------------- Configuration -----------------------

OUTPUT_ROOT = Path("/kaggle/working/overture_data")

# None discovers the latest release once at startup.
# Set a release string when you want to reproduce an earlier extraction.
RELEASE = None

# Buffered acquisition rectangle: [west, south, east, north].
# Includes neighbouring territory to preserve roads near India's borders.
BBOX = [67.0, 5.0, 99.0, 38.0]

MEMORY_LIMIT = "4GB"
THREADS = 4

CATALOG_URL = "https://stac.overturemaps.org/catalog.json"


# -------------------------- Helpers --------------------------

def sql_string(value):
    """Safely encode a string as a SQL literal."""
    return "'" + str(value).replace("'", "''") + "'"


def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".partial")
    temporary.write_text(
        json.dumps(value, indent=2, default=str),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))


def fetch_json(url):
    response = session.get(url, timeout=(20, 120))
    response.raise_for_status()
    return response.json()


west, south, east, north = map(float, BBOX)

if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
    raise ValueError("Invalid bounding box.")


# -------------------- Resolve and record release --------------------

root_catalog = fetch_json(CATALOG_URL)

if RELEASE is None:
    RELEASE = root_catalog["latest"]

# Prefer the catalog's own release link when available.
release_catalog_url = urljoin(
    CATALOG_URL, f"{RELEASE}/catalog.json"
)

for link in root_catalog.get("links", []):
    href = link.get("href", "")
    if (
        link.get("rel") == "child"
        and f"/{RELEASE}/" in urljoin(CATALOG_URL, href)
    ):
        release_catalog_url = urljoin(CATALOG_URL, href)
        break

release_catalog = fetch_json(release_catalog_url)

started = datetime.now(timezone.utc)
run_id = started.strftime("%Y%m%dT%H%M%S_%fZ")

# A separate directory per run prevents mixing different snapshots.
output = OUTPUT_ROOT / f"{RELEASE}_{run_id}"
output.mkdir(parents=True, exist_ok=False)

temporary_directory = output / "_duckdb_temp"
temporary_directory.mkdir()

write_json(output / "root_catalog.json", root_catalog)
write_json(output / "release_catalog.json", release_catalog)

manifest = {
    "status": "running",
    "release": RELEASE,
    "schema_version": release_catalog.get("schema:version"),
    "started_utc": started.isoformat(),
    "bbox": BBOX,
    "selection": "Segment bounding boxes intersect acquisition rectangle",
    "geometry": "Complete geometries; no clipping or simplification",
    "subtype_filter": None,
    "columns": "All source columns",
    "duckdb_version": duckdb.__version__,
}

write_json(output / "manifest.json", manifest)

print(f"Release: {RELEASE}", flush=True)
print(f"Output:  {output}", flush=True)


# ----------------------- DuckDB setup -----------------------

con = duckdb.connect()

try:
    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")
    con.execute("INSTALL spatial")
    con.execute("LOAD spatial")

    con.execute("SET s3_region = 'us-west-2'")
    con.execute(f"SET memory_limit = {sql_string(MEMORY_LIMIT)}")
    con.execute(f"SET threads = {int(THREADS)}")
    con.execute("SET preserve_insertion_order = false")
    con.execute(
        f"SET temp_directory = {sql_string(temporary_directory)}"
    )

    base = (
        f"s3://overturemaps-us-west-2/release/{RELEASE}"
        "/theme=transportation"
    )

    segment_source = (
        f"read_parquet({sql_string(base + '/type=segment/*.parquet')}, "
        "hive_partitioning=true)"
    )

    connector_source = (
        f"read_parquet({sql_string(base + '/type=connector/*.parquet')}, "
        "hive_partitioning=true)"
    )

    manifest["source_base"] = base

    def export_parquet(query, filename):
        """Write atomically; incomplete files retain the .partial suffix."""
        destination = output / filename
        partial = output / (filename + ".partial")

        con.execute(f"""
            COPY ({query})
            TO {sql_string(partial)}
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD,
                ROW_GROUP_SIZE 65536
            )
        """)

        partial.replace(destination)
        return destination

    def scalar(query):
        return con.execute(query).fetchone()[0]

    # ----------------------- Segments -----------------------

    print("Downloading all transportation segments in the region...", flush=True)

    # Keep records with missing bbox values rather than silently discarding them.
    # Normally official records provide these bounds.
    segment_file = export_parquet(
        f"""
        SELECT *
        FROM {segment_source}
        WHERE
            bbox.xmin IS NULL OR bbox.ymin IS NULL
            OR bbox.xmax IS NULL OR bbox.ymax IS NULL
            OR (
                bbox.xmin <= {east}
                AND bbox.xmax >= {west}
                AND bbox.ymin <= {north}
                AND bbox.ymax >= {south}
            )
        """,
        "segments.parquet",
    )

    con.execute(f"""
        CREATE TEMP VIEW segments AS
        SELECT * FROM read_parquet({sql_string(segment_file)})
    """)

    segment_count = scalar("SELECT count(*) FROM segments")

    if segment_count == 0:
        raise RuntimeError("No segments downloaded; check release and region.")

    print(f"Segments downloaded: {segment_count:,}", flush=True)

    # Record the local schema for future parser development.
    manifest["segment_schema"] = [
        {"name": row[0], "type": row[1]}
        for row in con.execute("DESCRIBE segments").fetchall()
    ]

    # Preserve the set of connector IDs actually needed by these segments.
    con.execute("""
        CREATE TEMP TABLE required_connectors AS
        SELECT DISTINCT ref.connector_id AS id
        FROM segments AS s
        CROSS JOIN UNNEST(s.connectors) AS u(ref)
        WHERE ref.connector_id IS NOT NULL
    """)

    # The full geometries can extend outside the original acquisition box.
    # Use their combined extent for the initial connector extraction.
    bounds = con.execute("""
        SELECT
            min(ST_XMin(geometry)),
            min(ST_YMin(geometry)),
            max(ST_XMax(geometry)),
            max(ST_YMax(geometry))
        FROM segments
    """).fetchone()

    if any(value is None for value in bounds):
        raise RuntimeError("Could not determine the segment geometry extent.")

    padding = 0.000001
    cw, cs, ce, cn = (
        bounds[0] - padding,
        bounds[1] - padding,
        bounds[2] + padding,
        bounds[3] + padding,
    )

    # ----------------------- Connectors -----------------------

    print("Downloading connectors over the complete segment extent...", flush=True)

    export_parquet(
        f"""
        SELECT *
        FROM {connector_source}
        WHERE
            bbox.xmin <= {ce}
            AND bbox.xmax >= {cw}
            AND bbox.ymin <= {cn}
            AND bbox.ymax >= {cs}
        """,
        "connectors_region.parquet",
    )

    con.execute(f"""
        CREATE TEMP VIEW initial_connectors AS
        SELECT *
        FROM read_parquet(
            {sql_string(output / 'connectors_region.parquet')}
        )
    """)

    con.execute("""
        CREATE TEMP TABLE missing_connectors AS
        SELECT r.id
        FROM required_connectors AS r
        ANTI JOIN initial_connectors AS c ON r.id = c.id
    """)

    missing_count = scalar("SELECT count(*) FROM missing_connectors")

    if missing_count:
        print(
            f"Fetching {missing_count:,} connectors by ID. "
            "This fallback may scan the global connector table.",
            flush=True,
        )

        export_parquet(
            f"""
            SELECT c.*
            FROM {connector_source} AS c
            SEMI JOIN missing_connectors AS m ON c.id = m.id
            """,
            "connectors_extra.parquet",
        )

    con.execute(f"""
        CREATE TEMP VIEW connectors AS
        SELECT *
        FROM read_parquet(
            {sql_string(output / 'connectors_*.parquet')}
        )
    """)

    # ----------------------- Validation -----------------------

    print("Checking downloaded records and connector references...", flush=True)

    stats = {}

    for table in ("segments", "connectors"):
        total, distinct_ids, null_ids = con.execute(f"""
            SELECT
                count(*),
                count(DISTINCT id),
                count(*) FILTER (WHERE id IS NULL)
            FROM {table}
        """).fetchone()

        stats[table] = total

        if null_ids or total != distinct_ids:
            raise RuntimeError(f"{table}: null or duplicate IDs found.")

        invalid_geometry = scalar(f"""
            SELECT count(*)
            FROM {table}
            WHERE geometry IS NULL OR NOT ST_IsValid(geometry)
        """)

        if invalid_geometry:
            raise RuntimeError(
                f"{table}: {invalid_geometry:,} null/invalid geometries."
            )

    bad_connector_lists = scalar("""
        SELECT count(*)
        FROM segments
        WHERE connectors IS NULL OR len(connectors) < 2
    """)

    null_connector_ids = scalar("""
        SELECT count(*)
        FROM segments AS s
        CROSS JOIN UNNEST(s.connectors) AS u(ref)
        WHERE ref.connector_id IS NULL
    """)

    unresolved = scalar("""
        SELECT count(*)
        FROM required_connectors AS r
        ANTI JOIN connectors AS c ON r.id = c.id
    """)

    if bad_connector_lists or null_connector_ids or unresolved:
        raise RuntimeError(
            "Connector validation failed: "
            f"bad lists={bad_connector_lists}, "
            f"null IDs={null_connector_ids}, "
            f"missing records={unresolved}"
        )

    stats["required_connectors"] = scalar(
        "SELECT count(*) FROM required_connectors"
    )
    stats["missing_connector_records"] = unresolved

    manifest["counts"] = stats
    manifest["subtype_counts"] = [
        {"subtype": row[0], "count": row[1]}
        for row in con.execute("""
            SELECT subtype, count(*)
            FROM segments
            GROUP BY subtype
            ORDER BY count(*) DESC
        """).fetchall()
    ]
    manifest["connector_schema"] = [
        {"name": row[0], "type": row[1]}
        for row in con.execute("DESCRIBE connectors").fetchall()
    ]

    # ----------------------- Final manifest -----------------------

    print("Computing file checksums...", flush=True)

    manifest["files"] = [
        {
            "name": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(output.glob("*.parquet"))
    ]

    manifest["status"] = "complete"
    manifest["finished_utc"] = datetime.now(timezone.utc).isoformat()

    write_json(output / "manifest.json", manifest)

    # This marker only exists after all checks and checksums succeed.
    (output / "_SUCCESS").write_text(
        f"{RELEASE}\n", encoding="utf-8"
    )

    print("\nDownload and validation complete.", flush=True)
    print(json.dumps(stats, indent=2), flush=True)
    print(f"\nSaved in: {output}", flush=True)

except Exception as error:
    manifest["status"] = "failed"
    manifest["error"] = str(error)
    write_json(output / "manifest.json", manifest)
    raise

finally:
    con.close()
    session.close()

Release: 2026-10-21.0
Output:  /kaggle/working/overture_data/2026-10-21.0_20261022T155257_952725Z


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))